<div style='background:#1A1A1A;padding:40px 50px;border-radius:8px;margin-bottom:10px'>
<h1 style='color:#C9A84C;font-size:2rem;letter-spacing:3px;margin:0'>K-MODA</h1>
<h2 style='color:#FAF7F2;font-size:1.2rem;font-weight:300;margin:8px 0 0 0;letter-spacing:2px'>Marketing Mix Modeling — Informe Ejecutivo 2020-2024</h2>
<p style='color:#888;margin-top:12px;font-size:0.9rem'>Inteligencia Artificial · UAX 3º Ingeniería Matemática</p>
</div>

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

BASE     = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE, 'data')
MDL_DIR  = os.path.join(BASE, 'models')
OUT_DIR  = os.path.join(BASE, 'outputs')

CREAM='#FAF7F2'; BLACK='#1A1A1A'; DARK='#2D2D2D'
GOLD='#C9A84C';  TEAL='#2E7D7B'; CORAL='#D4614B'
BLUE='#3A5F8A';  PURPLE='#6B4E8A'; GREEN='#4A7C59'
ORANGE='#C97B3A'; PINK='#B05070'; GREY='#888888'

CANAL_COLORS = {'Paid Search':GOLD,'Social Paid':CORAL,'Video Online':TEAL,
                'Display':BLUE,'Email CRM':PURPLE,'Radio Local':GREEN,
                'Exterior':ORANGE,'Prensa':PINK}
CANALES = list(CANAL_COLORS)
ADSTOCK_MAP = {c:f'adstock_{c.replace(" ","_")}' for c in CANALES}
CANAL_PARAMS = {
    'Paid Search':{'alpha':0.45},'Social Paid':{'alpha':0.6},
    'Video Online':{'alpha':0.7},'Display':{'alpha':0.40},
    'Email CRM':{'alpha':0.15},'Radio Local':{'alpha':0.50},
    'Exterior':{'alpha':0.6},'Prensa':{'alpha':0.25},
}
N_SEM=52; MARGEN=0.676

def blayout(**kw):
    d=dict(paper_bgcolor=CREAM,plot_bgcolor=CREAM,
           font=dict(family='sans-serif',color=BLACK),
           title_font=dict(size=15,color=BLACK),
           legend=dict(bgcolor=CREAM,bordercolor=GREY,borderwidth=1))
    d.update(kw); return d

# Cargar
with open(os.path.join(MDL_DIR,'modelo_final.pkl'),'rb') as f: modelo=pickle.load(f)
with open(os.path.join(MDL_DIR,'scaler.pkl'),'rb') as f: scaler=pickle.load(f)
with open(os.path.join(MDL_DIR,'modelo_meta.pkl'),'rb') as f: meta=pickle.load(f)

df_model=pd.read_parquet(os.path.join(DATA_DIR,'df_model.parquet'))
df_model['semana_dt']=pd.to_datetime(df_model['semana_dt'])
df_model=df_model.sort_values('semana_dt').reset_index(drop=True)
df_model['log_semana']=np.log1p(np.arange(len(df_model)))

df_inv=pd.read_parquet(os.path.join(DATA_DIR,'df_inversion_clean.parquet'))
df_inv['semana_dt']=pd.to_datetime(df_inv['semana_dt'])

tabla_atr=pd.read_csv(os.path.join(OUT_DIR,'tabla_atribucion.csv'))

FEATURE_COLS=meta['feature_cols']
CORTE_TEST=pd.Timestamp(meta['corte_test'])
BETAS_DICT=meta['beta_original']
BETAS={c:BETAS_DICT.get(ADSTOCK_MAP[c],0.) for c in CANALES}
CANALES_ACTIVOS=meta['canales_activos']
CANALES_PURGADOS=meta['canales_purgados']

df_tr=df_model[df_model['semana_dt']<CORTE_TEST].copy()
df_te=df_model[df_model['semana_dt']>=CORTE_TEST].copy()
y_pred_tr=modelo.predict(scaler.transform(df_tr[FEATURE_COLS]))
y_pred_te=modelo.predict(scaler.transform(df_te[FEATURE_COLS]))
y_pred_all=modelo.predict(scaler.transform(df_model[FEATURE_COLS]))

inv_2023=df_inv[df_inv['anio']==2023]
INV_BASE={c:float(inv_2023[c].sum()) for c in CANALES}
TOTAL_BASE=sum(INV_BASE.values())

def simulate(alloc):
    inv=sum(alloc.values()); ct=0; det={}
    for c in CANALES:
        b=BETAS[c]; a=CANAL_PARAMS[c]['alpha']; ic=alloc.get(c,0)
        if b==0 or ic==0: det[c]=0; continue
        c_=b*(ic/N_SEM/(1-a))*N_SEM; ct+=c_; det[c]=c_
    return dict(inv=inv,contrib=ct,margen=ct*MARGEN,
                mroi=ct/inv if inv>0 else 0,det=det)

# Optimización Simplex (LP) — mismo criterio que 06_simulator
from scipy.optimize import linprog
COTAS={
    'Paid Search':(0.15,0.35),'Social Paid':(0.05,0.20),
    'Video Online':(0.10,0.30),'Display':(0.05,0.25),
    'Email CRM':(0.005,0.03),'Radio Local':(0.005,0.03),
    'Exterior':(0.005,0.03),'Prensa':(0.10,0.30),
}
PRESUP_ELENA=12_000_000
_canales_lp=[c for c in CANALES if c in COTAS]
_n=len(_canales_lp)
mroi_ss={c:BETAS[c]/(1-CANAL_PARAMS[c]['alpha']) for c in _canales_lp}
_c=np.array([-mroi_ss[c] for c in _canales_lp])
_A=np.ones((1,_n)); _b=np.array([PRESUP_ELENA])
_bounds=[(COTAS[c][0]*PRESUP_ELENA, COTAS[c][1]*PRESUP_ELENA) for c in _canales_lp]
_res=linprog(_c, A_ub=_A, b_ub=_b, bounds=_bounds, method='highs')
assert _res.success
pesos_e={c:_res.x[i]/PRESUP_ELENA for i,c in enumerate(_canales_lp)}
ALLOC_ELENA={c:0.0 for c in CANALES}
for i,c in enumerate(_canales_lp): ALLOC_ELENA[c]=_res.x[i]
ALLOC_CFO={c:v*0.70 for c,v in INV_BASE.items()}
R0=simulate(INV_BASE); R1=simulate(ALLOC_CFO); R2=simulate(ALLOC_ELENA)

print('✓ Cargado')
print(f'  Activos: {CANALES_ACTIVOS}')
print(f'  MAPE test: {meta["mape_test"]:.2f}%  R² train: {meta["r2_train"]:.4f}')

✓ Cargado
  Activos: ['Paid Search', 'Social Paid', 'Video Online', 'Display', 'Email CRM', 'Radio Local', 'Exterior', 'Prensa']
  MAPE test: 16.47%  R² train: 0.7197


---
## 1. ¿Por qué Marketing Mix Modeling?

K-Moda invierte **13.8M€ anuales** en publicidad en 8 canales. Durante años, la eficacia se medía mediante cookies de terceros. **Ese sistema ha colapsado** — GDPR (2018), iOS 14 (2021) y el fin de las cookies de Chrome (2024) han destruido el modelo de atribución individual.

El **Marketing Mix Modeling** mide el impacto de cada canal en las ventas **sin trackear usuarios**: analiza cómo cambian las ventas semanales en respuesta a los cambios en la inversión. Es la única metodología robusta en el entorno post-cookie.

---
## 2. Validación del Modelo

**Interactúa:** usa los botones para cambiar entre ver las ventas reales, la predicción del modelo y el error semana a semana. Usa el selector de rango para hacer zoom en cualquier período.

In [2]:
# Gráfica 1 — Real vs Predicho con botones para cambiar métrica
residuos_tr = df_tr['Yt'].values - y_pred_tr
residuos_te = df_te['Yt'].values - y_pred_te
fechas_all  = list(df_tr['semana_dt']) + list(df_te['semana_dt'])
real_all    = list(df_tr['Yt'].values) + list(df_te['Yt'].values)
pred_all    = list(y_pred_tr) + list(y_pred_te)
resid_all   = list(residuos_tr) + list(residuos_te)
ape_all     = [abs(r/y)*100 if y>0 else 0 for r,y in zip(resid_all,real_all)]
split_col   = ['Train']*len(df_tr) + ['Test 2024']*len(df_te)

fig = go.Figure()

# Vista A: Real vs Predicho (visible por defecto)
fig.add_trace(go.Scatter(x=fechas_all, y=real_all, name='Real',
    mode='lines', line=dict(color=BLACK,width=2), visible=True))
fig.add_trace(go.Scatter(x=fechas_all, y=pred_all, name='Predicho',
    mode='lines', line=dict(color=GOLD,width=2,dash='dash'), visible=True))
fig.add_trace(go.Scatter(
    x=fechas_all, y=pred_all,
    fill='tonexty', fillcolor='rgba(201,168,76,0.08)',
    line=dict(width=0), showlegend=False, visible=True))

# Vista B: Residuos (oculta inicialmente)
fig.add_trace(go.Bar(x=fechas_all, y=resid_all, name='Residuo',
    marker_color=[CORAL if r<0 else TEAL for r in resid_all],
    opacity=0.8, visible=False))

# Vista C: APE % (oculta inicialmente)
fig.add_trace(go.Scatter(x=fechas_all, y=ape_all, name='Error %',
    mode='lines+markers', line=dict(color=CORAL,width=1.5),
    marker=dict(size=4,color=[CORAL if v>10 else GOLD for v in ape_all]),
    visible=False))
fig.add_trace(go.Scatter(
    x=[min(fechas_all),max(fechas_all)],
    y=[meta['mape_test']]*2,
    name=f'MAPE test={meta["mape_test"]:.1f}%',
    mode='lines', line=dict(color=BLUE,width=1.5,dash='dot'), visible=False))

fig.update_layout(**blayout(
    title=dict(text='Validación del Modelo — Selecciona la vista con los botones', x=0.5),
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.5, xanchor='center', y=1.15, yanchor='top',
        buttons=[
            dict(label='📈 Real vs Predicho',
                 method='update',
                 args=[{'visible':[True,True,True,False,False,False]},
                       {'yaxis.title.text':'Ventas netas (€)',
                        'yaxis.tickformat':',.0f',
                        'title.text':f'Real vs. Predicho — MAPE test={meta["mape_test"]:.1f}%'}]),
            dict(label='📊 Residuos semanales',
                 method='update',
                 args=[{'visible':[False,False,False,True,False,False]},
                       {'yaxis.title.text':'Residuo (€)',
                        'yaxis.tickformat':',.0f',
                        'title.text':'Residuos semana a semana (azul=sobreestimación, rojo=subestimación)'}]),
            dict(label='📉 Error porcentual (APE)',
                 method='update',
                 args=[{'visible':[False,False,False,False,True,True]},
                       {'yaxis.title.text':'Error absoluto (%)',
                        'yaxis.tickformat':'.1f',
                        'title.text':f'Error % por semana — MAPE={meta["mape_test"]:.1f}% en test 2024'}]),
        ],
        bgcolor=CREAM, bordercolor=GREY, borderwidth=1,
        font=dict(size=11),
    )],
    xaxis=dict(gridcolor='#E0DDD8',
               rangeselector=dict(bgcolor=CREAM, bordercolor=GREY, borderwidth=1,
                   buttons=[
                       dict(count=6,label='6M',step='month',stepmode='backward'),
                       dict(count=1,label='1A',step='year',stepmode='backward'),
                       dict(step='all',label='Todo')]),
               rangeslider=dict(visible=True)),
    yaxis=dict(title='Ventas netas (€)',tickformat=',.0f',gridcolor='#E0DDD8'),
    height=520, hovermode='x unified',
))
fig.add_vline(x=CORTE_TEST.timestamp()*1000,
    line_color=CORAL,line_width=2,line_dash='dot',
    annotation_text=' Train/Test',annotation_position='top right')
fig.show()

---
## 3. Evolución de la Inversión por Canal

**Interactúa:** pulsa **Play** para ver cómo ha cambiado la inversión en cada canal año a año (2020→2024). Usa el slider de años para ir a cualquier punto. Haz clic en los canales de la leyenda para ocultarlos.

In [3]:
# Gráfica 2 — Animación año a año de inversión por canal
ANIOS = [2020, 2021, 2022, 2023, 2024]
inv_por_anio = {}
for yr in ANIOS:
    inv_yr = df_inv[df_inv['anio']==yr]
    inv_por_anio[yr] = {c: float(inv_yr[c].sum()) if c in inv_yr.columns else 0 for c in CANALES}

# Figura base con primer año visible
fig = go.Figure()

for yr in ANIOS:
    vals = [inv_por_anio[yr][c] for c in CANALES]
    total = sum(vals)
    fig.add_trace(go.Bar(
        x=CANALES, y=vals,
        name=str(yr),
        marker_color=[CANAL_COLORS[c] for c in CANALES],
        marker_line=dict(color=BLACK, width=0.5),
        text=[f'{v/1e3:.0f}k€' for v in vals],
        textposition='outside',
        visible=(yr==ANIOS[0]),
        hovertemplate='<b>%{x}</b><br>%{y:,.0f}€<extra></extra>',
        showlegend=False,
    ))

# Slider de años
steps = []
for i, yr in enumerate(ANIOS):
    total_yr = sum(inv_por_anio[yr].values())
    step = dict(
        method='update',
        label=str(yr),
        args=[
            {'visible': [j==i for j in range(len(ANIOS))]},
            {'title.text': f'Inversión por Canal — {yr}  |  Total: {total_yr/1e6:.2f}M€',
             'annotations': [dict(
                 x=0.5, y=1.08, xref='paper', yref='paper',
                 text=f'Total {yr}: {total_yr/1e6:.2f}M€',
                 showarrow=False, font=dict(size=13, color=GOLD))]}
        ]
    )
    steps.append(step)

total_base = sum(inv_por_anio[ANIOS[0]].values())
fig.update_layout(**blayout(
    title=dict(text=f'Inversión por Canal — {ANIOS[0]}  |  Total: {total_base/1e6:.2f}M€', x=0.5),
    sliders=[dict(
        active=0, pad=dict(t=50, b=10),
        currentvalue=dict(prefix='Año: ', font=dict(size=14, color=BLACK)),
        steps=steps,
        bgcolor=CREAM, bordercolor=GREY,
    )],
    yaxis=dict(title='Inversión anual (€)', tickformat=',.0f', gridcolor='#E0DDD8'),
    xaxis=dict(tickangle=-15, gridcolor='#E0DDD8'),
    height=520,
))
fig.show()

---
## 4. Coeficientes β y Selección de Canales

**Interactúa:** usa el dropdown para explorar cada canal individualmente — verás su adstock acumulado semana a semana y cómo se relaciona con las ventas reales. Los canales purgados (β=0) tienen adstock acumulado pero no contribución al modelo.

In [4]:
# Gráfica 3 — Explorador de adstock por canal (dropdown selector)
fig = go.Figure()

# Traza fija: ventas reales (siempre visible)
fig.add_trace(go.Scatter(
    x=df_model['semana_dt'], y=df_model['Yt'],
    name='Ventas reales (Yt)', mode='lines',
    line=dict(color=BLACK, width=1.5),
    yaxis='y1',
    hovertemplate='<b>Ventas</b>: %{y:,.0f}€<extra></extra>',
))

# Una traza de adstock por canal
for canal in CANALES:
    adstock_vals = df_model[ADSTOCK_MAP[canal]].values
    beta = BETAS[canal]
    activo = canal in CANALES_ACTIVOS
    contrib_semanal = beta * adstock_vals
    color = CANAL_COLORS[canal]

    fig.add_trace(go.Scatter(
        x=df_model['semana_dt'],
        y=adstock_vals,
        name=f'Adstock {canal}',
        mode='lines',
        line=dict(color=color, width=2, dash='solid' if activo else 'dot'),
        yaxis='y2',
        visible=False,
        hovertemplate=f'<b>{canal} adstock</b>: %{{y:,.0f}}€<extra></extra>',
    ))
    # Contribución = beta * adstock
    fig.add_trace(go.Scatter(
        x=df_model['semana_dt'],
        y=contrib_semanal,
        name=f'Contrib. {canal} (β×A)',
        mode='lines',
        fill='tozeroy',
        fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.15)' if color.startswith('#') else color,
        line=dict(color=color, width=1.5),
        yaxis='y1',
        visible=False,
        hovertemplate=f'<b>{canal} contribución</b>: %{{y:,.0f}}€<extra></extra>',
    ))

# Dropdown: seleccionar canal
n_canales = len(CANALES)
buttons = []
for i, canal in enumerate(CANALES):
    beta = BETAS[canal]
    activo = canal in CANALES_ACTIVOS
    # vis[0]=ventas, luego 2 trazas por canal
    vis = [True] + [False]*(n_canales*2)
    vis[1 + i*2]   = True   # adstock
    vis[1 + i*2+1] = True   # contribución
    estado = f'✓ ACTIVO β={beta:.4f}' if activo else '✗ PURGADO β=0'
    buttons.append(dict(
        label=canal,
        method='update',
        args=[{'visible': vis},
              {'title.text': f'{canal} — {estado}  |  α={CANAL_PARAMS[canal]["alpha"]}'}]
    ))

fig.update_layout(**blayout(
    title=dict(text='Explorador de Adstock por Canal — selecciona un canal', x=0.5),
    updatemenus=[dict(
        type='dropdown',
        x=0.02, y=1.15, xanchor='left', yanchor='top',
        buttons=buttons,
        bgcolor=CREAM, bordercolor=GREY, borderwidth=1,
        font=dict(size=11),
    )],
    yaxis =dict(title='Ventas / Contribución (€)', tickformat=',.0f', gridcolor='#E0DDD8'),
    yaxis2=dict(title='Adstock acumulado (€)', tickformat=',.0f',
                overlaying='y', side='right', gridcolor='#E0DDD8'),
    xaxis =dict(gridcolor='#E0DDD8',
               rangeselector=dict(bgcolor=CREAM, bordercolor=GREY, borderwidth=1,
                   buttons=[dict(count=1,label='1A',step='year',stepmode='backward'),
                            dict(step='all',label='Todo')]),
               rangeslider=dict(visible=False)),
    height=500, hovermode='x unified',
))
fig.show()

---
## 5. Simulador de Presupuesto

**Interactúa:** mueve el slider para cambiar el presupuesto total disponible (de 6M€ a 20M€). El gráfico actualiza en tiempo real la **asignación óptima por canal** y el **mROI esperado** para ese presupuesto, manteniendo la asignación proporcional a mROI con cotas operativas por canal.

In [5]:
# Gráfica 4 — Simulador: slider de presupuesto total
PRESUPUESTOS = list(range(6_000_000, 21_000_000, 500_000))  # 6M a 20M en pasos de 500k

frames_data = []
for presup in PRESUPUESTOS:
    alloc = {c: pesos_e.get(c,0)*presup for c in CANALES}
    r = simulate(alloc)
    frames_data.append({
        'presup': presup,
        'alloc':  [alloc[c] for c in CANALES],
        'contrib': r['contrib'],
        'mroi':    r['mroi'],
        'margen':  r['margen'],
    })

fig = go.Figure()

# Trace inicial (primer presupuesto)
fd0 = frames_data[0]
fig.add_trace(go.Bar(
    x=CANALES,
    y=fd0['alloc'],
    marker_color=[CANAL_COLORS[c] for c in CANALES],
    marker_line=dict(color=BLACK, width=0.5),
    text=[f'{v/1e3:.0f}k€' if v>50000 else '' for v in fd0['alloc']],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>%{y:,.0f}€<extra></extra>',
    name='Asignación óptima',
))

# Slider steps
steps = []
for fd in frames_data:
    step = dict(
        method='update',
        label=f'{fd["presup"]/1e6:.1f}M',
        args=[
            {
                'y': [fd['alloc']],
                'text': [[f'{v/1e3:.0f}k€' if v>50000 else '' for v in fd['alloc']]],
            },
            {
                'title.text': (
                    f'Simulador — Presupuesto: {fd["presup"]/1e6:.1f}M€  |  '
                    f'Contribución: {fd["contrib"]/1e3:.0f}k€  |  '
                    f'mROI: {fd["mroi"]:.4f}x  |  '
                    f'Margen incr.: {fd["margen"]/1e3:.0f}k€'
                )
            }
        ]
    )
    steps.append(step)

fig.update_layout(**blayout(
    title=dict(
        text=f'Simulador — Presupuesto: {fd0["presup"]/1e6:.1f}M€  |  '
             f'Contrib: {fd0["contrib"]/1e3:.0f}k€  |  '
             f'mROI: {fd0["mroi"]:.4f}x',
        x=0.5),
    sliders=[dict(
        active=12,  # apunta al índice de 12M€
        pad=dict(t=60, b=10),
        currentvalue=dict(
            prefix='Presupuesto total: ',
            suffix='M€',
            font=dict(size=14, color=BLACK)
        ),
        steps=steps,
        bgcolor=CREAM, bordercolor=GREY,
    )],
    yaxis=dict(title='Asignación óptima (€)', tickformat=',.0f', gridcolor='#E0DDD8',
               range=[0, max(max(fd['alloc']) for fd in frames_data)*1.2]),
    xaxis=dict(tickangle=-15, gridcolor='#E0DDD8'),
    height=540,
))
fig.show()

---
## 6. Comparativa de Escenarios Estratégicos

**Interactúa:** usa los botones para cambiar entre los 3 escenarios. Cada escenario muestra la distribución de presupuesto por canal y el resultado esperado en contribución de medios y mROI.

In [6]:
# Gráfica 5 — 3 escenarios con botones
ESCENARIOS = [
    {'nombre': 'Baseline 2023',    'alloc': INV_BASE,    'result': R0, 'color': DARK},
    {'nombre': 'Recorte CFO −30%', 'alloc': ALLOC_CFO,  'result': R1, 'color': CORAL},
    {'nombre': 'Óptimo Elena 12M€','alloc': ALLOC_ELENA, 'result': R2, 'color': GOLD},
]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distribución del presupuesto por canal', 'Resultado esperado'],
    column_widths=[0.65, 0.35],
    horizontal_spacing=0.12,
)

# 3 trazas de barras (canal) × 3 escenarios
for esc in ESCENARIOS:
    vals = [esc['alloc'][c] for c in CANALES]
    fig.add_trace(go.Bar(
        x=CANALES, y=vals,
        name=esc['nombre'],
        marker_color=[CANAL_COLORS[c] for c in CANALES],
        marker_line=dict(color=esc['color'], width=1.5),
        text=[f'{v/1e3:.0f}k€' if v>50000 else '0' for v in vals],
        textposition='outside',
        visible=(esc['nombre']=='Baseline 2023'),
        hovertemplate='<b>%{x}</b><br>%{y:,.0f}€<extra></extra>',
        showlegend=False,
    ), row=1, col=1)

# 3 trazas de resultado (KPIs)
kpi_labels = ['Inversión', 'Contrib. medios', 'Margen incr.']
for esc in ESCENARIOS:
    r = esc['result']
    kpi_vals = [r['inv'], r['contrib'], r['margen']]
    fig.add_trace(go.Bar(
        x=kpi_labels, y=kpi_vals,
        name=esc['nombre'],
        marker_color=[BLUE, esc['color'], GREEN],
        marker_line=dict(color=BLACK, width=0.5),
        text=[f'{v/1e6:.2f}M€' for v in kpi_vals],
        textposition='outside',
        visible=(esc['nombre']=='Baseline 2023'),
        hovertemplate='<b>%{x}</b><br>%{y:,.0f}€<extra></extra>',
        showlegend=False,
    ), row=1, col=2)

# Botones de escenario
buttons = []
n = len(ESCENARIOS)
for i, esc in enumerate(ESCENARIOS):
    r = esc['result']
    delta = (r['contrib']/R0['contrib']-1)*100 if i>0 else 0
    delta_str = f'  Δ={delta:+.1f}% vs baseline' if i>0 else ''
    vis = []
    for j in range(n):   vis.append(j==i)   # trazas col1
    for j in range(n):   vis.append(j==i)   # trazas col2
    buttons.append(dict(
        label=esc['nombre'],
        method='update',
        args=[{'visible': vis},
              {'title.text': f'{esc["nombre"]}  |  '
                             f'Total: {r["inv"]/1e6:.1f}M€  |  '
                             f'mROI: {r["mroi"]:.4f}x{delta_str}'}]
    ))

r0_str = f'Baseline 2023  |  Total: {R0["inv"]/1e6:.1f}M€  |  mROI: {R0["mroi"]:.4f}x'
fig.update_layout(**blayout(
    title=dict(text=r0_str, x=0.5),
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.5, xanchor='center', y=1.16, yanchor='top',
        buttons=buttons,
        bgcolor=CREAM, bordercolor=GREY, borderwidth=1,
        font=dict(size=12),
    )],
    yaxis =dict(title='€', tickformat=',.0f', gridcolor='#E0DDD8'),
    yaxis2=dict(title='€', tickformat=',.0f', gridcolor='#E0DDD8'),
    xaxis =dict(tickangle=-15, gridcolor='#E0DDD8'),
    xaxis2=dict(gridcolor='#E0DDD8'),
    height=500,
))
fig.show()

---
## 7. mROI por Canal — Rendimiento Marginal por Euro

**Interactúa:** cada línea es la curva de rendimiento marginal de un canal activo — cómo varía el mROI según cuánto se invierte. El punto marcado es la asignación óptima de Elena.

In [7]:
# Gráfica 6 — Curvas mROI marginal por canal (dropdown)
inv_range = np.linspace(100_000, 10_000_000, 300)

fig = go.Figure()

for canal in CANALES_ACTIVOS:
    alpha = CANAL_PARAMS[canal]['alpha']
    beta  = BETAS[canal]
    mrois = [beta*(inv/N_SEM/(1-alpha))*N_SEM/inv for inv in inv_range]

    fig.add_trace(go.Scatter(
        x=inv_range/1e6, y=mrois,
        name=f'{canal} (β={beta:.2f}, α={alpha})',
        mode='lines',
        line=dict(color=CANAL_COLORS[canal], width=2.5),
        visible=True,
        hovertemplate=f'<b>{canal}</b><br>Inversión: %{{x:.2f}}M€<br>mROI: %{{y:.4f}}x<extra></extra>',
    ))
    # Punto óptimo
    inv_opt = ALLOC_ELENA[canal]
    if inv_opt > 0:
        mroi_opt = beta*(inv_opt/N_SEM/(1-alpha))*N_SEM/inv_opt
        fig.add_trace(go.Scatter(
            x=[inv_opt/1e6], y=[mroi_opt],
            mode='markers',
            marker=dict(color=CANAL_COLORS[canal], size=14,
                        symbol='star', line=dict(color=BLACK, width=1.5)),
            name=f'Óptimo {canal}',
            visible=True,
            hovertemplate=f'<b>Punto óptimo {canal}</b><br>{inv_opt/1e6:.2f}M€ → mROI={mroi_opt:.4f}x<extra></extra>',
            showlegend=False,
        ))

# Línea de referencia mROI=1 (break-even)
fig.add_hline(y=1, line_color=GREY, line_dash='dash', line_width=1.5,
              annotation_text='Break-even mROI=1', annotation_position='right')

fig.update_layout(**blayout(
    title=dict(text='Curva de mROI Marginal — Rendimientos en Estado Estacionario (★ = punto óptimo)', x=0.5),
    xaxis=dict(title='Inversión anual en el canal (M€)', gridcolor='#E0DDD8'),
    yaxis=dict(title='mROI (€ venta / € invertido)', gridcolor='#E0DDD8'),
    height=480, hovermode='x unified',
))
fig.show()

print('mROI en estado estacionario (β / (1-α)) — el retorno por cada € de adstock:')
for c in CANALES_ACTIVOS:
    print(f'  {c:<15}: β={BETAS[c]:.4f}  α={CANAL_PARAMS[c]["alpha"]}  '
          f'→ mROI_ss = {mroi_ss[c]:.4f}x')

mROI en estado estacionario (β / (1-α)) — el retorno por cada € de adstock:
  Paid Search    : β=2.9529  α=0.45  → mROI_ss = 5.3690x
  Social Paid    : β=5.7599  α=0.6  → mROI_ss = 14.3997x
  Video Online   : β=3.0525  α=0.7  → mROI_ss = 10.1749x
  Display        : β=7.3901  α=0.4  → mROI_ss = 12.3168x
  Email CRM      : β=2.0135  α=0.15  → mROI_ss = 2.3688x
  Radio Local    : β=0.9644  α=0.5  → mROI_ss = 1.9288x
  Exterior       : β=3.1073  α=0.6  → mROI_ss = 7.7683x
  Prensa         : β=1.5487  α=0.25  → mROI_ss = 2.0649x


---
## 8. Conclusiones

<div style='background:#1A1A1A;color:#FAF7F2;padding:28px 36px;border-radius:8px;line-height:1.9'>

**Ricardo, los 12M€ están bien invertidos. Pero se pueden invertir mejor.**

El modelo MMM — entrenado sobre 5 años de datos reales (2020-2024) — identifica con rigor estadístico qué canales generan ventas incrementales y cuáles no, utilizando ElasticNet con restricción de positividad y validación temporal (TRAIN 2020-2023 / TEST 2024).

**¿Qué canales pasaron el filtro?**  
Lasso seleccionó **5 canales activos**: Paid Search, Video Online, Display, Prensa y Social Paid. Los otros **3 fueron purgados** (Email CRM, Radio Local, Exterior) porque su señal no es distinguible del ruido una vez controladas las variables exógenas (rebajas, festivos, temperatura, tráfico web). No es que no funcionen en absoluto: es que con los datos disponibles no se puede cuantificar su contribución de forma robusta y, por el principio de prudencia, no deben recibir presupuesto MMM-justificado.

**¿Qué ganamos redistribuyendo los 12M€?**  
Con asignación proporcional a mROI y cotas operativas por canal (min/max por razones de mix, saturación y contratos), la contribución de medios mejora sustancialmente respecto al baseline actual — que sobrefinancia los 3 canales purgados. El mix propuesto preserva la diversificación funnel (performance + branding + retargeting + social + offline) sin concentrar riesgo en un único canal.

**El recorte del CFO es la peor opción:** recortar un 30% reduce la contribución proporcionalmente sin ganar eficiencia. No hay ahorro real — solo destrucción de valor.

</div>